In [ ]:
!pip install -U transformers datasets accelerate sentencepiece --quiet

# imports

In [ ]:
import os
import gc
import random
import shutil
import time
from dataclasses import dataclass
from typing import Dict, List, Optional
import math
import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR
from torch.optim.optimizer import Optimizer

from datasets import load_dataset, Dataset
from transformers import (
    T5ForConditionalGeneration,
    T5TokenizerFast,
    DataCollatorForSeq2Seq,
)
from sentence_transformers import SentenceTransformer

from peft import LoraConfig, TaskType, get_peft_model, PeftModel

try:
    from safetensors.torch import load_file as safe_load, save_file as safe_save
except Exception:
    safe_load, safe_save = None, None

from transformers.utils import logging
logging.set_verbosity_error()   # hides WARNING/INFO messages (incl. load reports)


In [ ]:
def cosine_lr(initial_lr: float,
              round_idx: int,
              total_rounds: int,
              eta_min: float = 0.0) -> float:
    round_idx = min(round_idx, total_rounds)
    cosine_decay = 0.5 * (1 + math.cos(math.pi * round_idx / total_rounds))
    return eta_min + (initial_lr - eta_min) * cosine_decay

In [ ]:
# from huggingface_hub import login
# login(os.environ["HF_TOKEN"])  # optional; public models/datasets need no token

In [ ]:
'''
src = "/kaggle/input/notebooks/aaserfawzy/non-last-exp-adam-base/client_state"
dst = "/kaggle/working/client_state"

for item in os.listdir(src):
    s = os.path.join(src, item)
    d = os.path.join(dst, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)

src = "/kaggle/input/notebooks/aaserfawzy/non-last-exp-adam-base/main_model"
dst = "/kaggle/working/main_model"

# Move the folder (cut)
shutil.copytree(src, dst, dirs_exist_ok=True)
'''

# Constants

In [ ]:
seed = 785638984
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

MODEL_NAME = "t5-small"
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 8
BATCH_SIZE = 64
EPOCHS = 1
LR = 6e-4
rounds = 50
Main_path = "./main_model"

numberOFClients = 30

OUTPUT_DIR = "t5_agnews_no_trainer"
MAIN_PATH_MODEL = Main_path
global_adapter_dir = Main_path
ADAPTER_BIN = "adapter_model.bin"
ADAPTER_SAFE = "adapter_model.safetensors"


In [ ]:
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
label2id = {v: k for k, v in id2label.items()}

# For robust matching of generated text
canonical_label = {
    "world": 0,
    "sports": 1,
    "business": 2,
    "sci/tech": 3,
    "science": 3,
    "technology": 3,
    "sci": 3,
    "tech": 3,
    "sci tech": 3,
    "sci-tech": 3,
}


In [ ]:
def normalize_label_text(s: str) -> Optional[int]:
    t = s.strip().lower()
    # keep only the first token or short phrase to avoid extra words
    t = t.replace(".", "").replace(",", "")
    # try exact first
    if t in canonical_label:
        return canonical_label[t]
    # try best-effort matching
    for key in canonical_label:
        if t.startswith(key):
            return canonical_label[key]
    return None

# model Import

In [ ]:
tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(device)
lora_config = LoraConfig(
    r=16,                       # rank
    lora_alpha=32,              # scaling
    lora_dropout=0.05,          # regularization
    bias="none",
    target_modules=["q", "k", "v", "o"],  # T5 attention proj weights
    task_type="SEQ_2_SEQ_LM"
)
model = get_peft_model(model, lora_config)
model.save_pretrained(MAIN_PATH_MODEL)
model.save_pretrained( "./main_model_pre")
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Data

In [ ]:
ds = load_dataset("fancyzhx/ag_news")  # splits: train/test
# build a validation split from train
train_valid = ds["train"].train_test_split(test_size=0.05, seed=seed,stratify_by_column= "label")
train_data = train_valid["train"]
validation_data = train_valid["test"]
test_data = ds["test"]



In [ ]:
label_counts = {}
for item in train_data:
    label = item['label']
    if label in label_counts:
        label_counts[label] += 1
    else:
        label_counts[label] = 1
print(f"Number of data points {len(train_data)}")
print("Number of data points per label in train_data:")
for label, count in label_counts.items():
    print(f"Label {label}: {count} data points")

In [ ]:
label_counts = {}
for item in validation_data:
    label = item['label']
    if label in label_counts:
        label_counts[label] += 1
    else:
        label_counts[label] = 1
print(f"Number of data points {len(validation_data)}")
print("Number of data points per label in test_dataset:")
for label, count in label_counts.items():
    print(f"Label {label}: {count} data points")

In [ ]:
label_counts = {}
for item in test_data:
    label = item['label']
    if label in label_counts:
        label_counts[label] += 1
    else:
        label_counts[label] = 1
print(f"Number of data points {len(test_data)}")
print("Number of data points per label in test_dataset:")
for label, count in label_counts.items():
    print(f"Label {label}: {count} data points")

In [ ]:
def build_text_batch(batch: Dict[str, List]) -> List[str]:
    if "text" in batch:
        return batch["text"]
    titles = batch.get("title", [""] * len(batch["label"]))
    descs = batch.get("description", [""] * len(batch["label"]))
    return [f"{t}. {d}".strip() for t, d in zip(titles, descs)]

def preprocess_fn(batch: Dict[str, List]) -> Dict[str, List]:
    texts = build_text_batch(batch)
    inputs = [f"classify news: {t}" for t in texts]
    targets = [id2label[int(l)] for l in batch["label"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )

    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=0)

cols = ["input_ids", "attention_mask", "labels"]

tokenized_valid = validation_data.map(preprocess_fn, batched=True, remove_columns=validation_data.column_names)
tokenized_test = test_data.map(preprocess_fn, batched=True, remove_columns=test_data.column_names)

tokenized_valid.set_format(type="torch", columns=cols)
tokenized_test.set_format(type="torch", columns=cols)

valid_loader   = DataLoader(tokenized_valid, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_loader   = DataLoader(tokenized_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

# functions

In [ ]:
def get_trained_client(setting,numberOfClient,numberParticpate):

    clients = list(range(numberOfClient))

    if setting == 1:
        return clients

    elif setting == 2:
        return  random.sample(clients, numberParticpate)

    elif setting == 3:
        count = random.randint(1, len(clients))
        return random.sample(clients, count)

    else:
        raise ValueError("no existing setting that can be used")

In [ ]:
@torch.no_grad()
def evaluate(model, loader: DataLoader, device):
    """
    Returns:
        accuracy: float
        nll: mean token-level negative log-likelihood
    """
    model.eval()
    correct = 0
    total = 0

    total_nll = 0.0
    total_tokens = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # ======== ACCURACY PART (UNCHANGED) ========
        gen = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=3,
            num_beams=1,
        )
        preds = tokenizer.batch_decode(gen, skip_special_tokens=True)
        gold = tokenizer.batch_decode(
            torch.where(
                batch["labels"] == -100,
                torch.full_like(batch["labels"], tokenizer.pad_token_id),
                batch["labels"],
            ),
            skip_special_tokens=True,
        )

        for p, g in zip(preds, gold):
            p_id = normalize_label_text(p)
            g_id = normalize_label_text(g)
            if g_id is None:
                continue
            total += 1
            if p_id == g_id:
                correct += 1

        # ======== NLL PART (ADDED, DOES NOT CHANGE ACC CODE) ========
        # Teacher forcing forward pass
        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        # out.loss is already MEAN CE over non-masked tokens
        batch_loss = out.loss.detach().item()

        # Count valid tokens (labels != -100)
        valid_tokens = (labels != -100).sum().item()

        total_nll += batch_loss * valid_tokens
        total_tokens += valid_tokens

    accuracy = correct / max(total, 1)
    nll = total_nll / max(total_tokens, 1)

    return accuracy, nll

In [ ]:
def _int_counts_from_simplex(prob, total, rng):
    """
    Convert simplex vector prob into integer counts that sum to total
    using floor + remainder distribution.
    """
    raw = prob * total
    counts = np.floor(raw).astype(int)
    rem = total - counts.sum()
    if rem > 0:
        frac = raw - counts
        # deterministic tie-breaking with rng noise
        order = np.argsort(-frac + 1e-12 * rng.random(len(frac)))
        counts[order[:rem]] += 1
    return counts

def dirichlet_partition_both(
    labels,
    num_clients: int = 10,
    alpha: float = 0.5,
    beta: float | None = None,
    prior: str = "uniform",      # you asked for uniform
    min_client_size: int = 1,
    seed: int = 42,
):

    rng = np.random.default_rng(seed)
    y = np.asarray(labels)
    N = len(y)

    # Map labels to 0..L-1
    classes = np.unique(y)
    L = len(classes)
    to_int = {c: i for i, c in enumerate(classes)}
    y_int = np.array([to_int[v] for v in y], dtype=int)

    # Prior p over labels (uniform per your requirement)
    if prior == "uniform":
        p = np.ones(L) / L
    elif prior == "empirical":
        cnt = np.bincount(y_int, minlength=L)
        p = cnt / cnt.sum()
    else:
        raise ValueError("prior must be 'uniform' or 'empirical'.")

    # ----- quantity sizes -----
    if beta is None:
        # balanced: same or very similar number of examples
        base = N // num_clients
        rem = N % num_clients
        sizes = [base + (1 if i < rem else 0) for i in range(num_clients)]
        z = None
    else:

        z = rng.dirichlet(np.ones(num_clients) * beta)
        sizes = _int_counts_from_simplex(z, N, rng).tolist()

        # enforce min_client_size if requested
        if min_client_size > 1:
            if min_client_size * num_clients > N:
                raise ValueError("min_client_size too large for total dataset size.")
            # raise small clients up to min by taking from larger ones
            deficit = sum(max(0, min_client_size - s) for s in sizes)
            if deficit > 0:
                order = np.argsort(-np.array(sizes))
                for i in order:
                    if deficit <= 0:
                        break
                    spare = sizes[i] - min_client_size
                    if spare > 0:
                        take = min(spare, deficit)
                        sizes[i] -= take
                        deficit -= take
                for j in range(num_clients):
                    need = max(0, min_client_size - sizes[j])
                    sizes[j] += need

            # re-fix sum exactly to N
            diff = N - sum(sizes)
            if diff > 0:
                for _ in range(diff):
                    sizes[rng.integers(num_clients)] += 1
            elif diff < 0:
                diff = -diff
                for _ in range(diff):
                    candidates = [i for i, s in enumerate(sizes) if s > min_client_size]
                    i = rng.choice(candidates)
                    sizes[i] -= 1

    # ----- label distributions -----

    conc = np.clip(alpha * p, 1e-12, None)  # ensure positive concentration
    q = rng.dirichlet(conc, size=num_clients)

    # Pools of indices per label (for sampling without replacement)
    pools = []
    for k in range(L):
        idxs = np.where(y_int == k)[0]
        rng.shuffle(idxs)
        pools.append(list(idxs))

    # Allocate without replacement according to q_j
    client_indices = []
    for j in range(num_clients):
        m = sizes[j]
        desired = rng.multinomial(m, q[j])

        assigned = []
        # take desired from each pool
        for k in range(L):
            take = min(desired[k], len(pools[k]))
            if take > 0:
                assigned.extend(pools[k][:take])
                pools[k] = pools[k][take:]

        while len(assigned) < m:
            avail = np.array([len(pools[k]) for k in range(L)])
            if avail.sum() == 0:
                break
            mask = (avail > 0).astype(float)
            probs = q[j] * mask
            probs = probs / probs.sum()
            k = rng.choice(L, p=probs)
            assigned.append(pools[k].pop())  # remove => without replacement

        rng.shuffle(assigned)
        client_indices.append(assigned)

    return client_indices, q, sizes, classes, z

In [ ]:
def make_client_shards(dataset, client_indices):
    """
    dataset: Hugging Face Dataset
    client_indices: list of list[int]
    returns: list[Dataset]
    """
    return [dataset.select(idxs) for idxs in client_indices]

In [ ]:
client_idxs, q, sizes, classes, z = dirichlet_partition_both(
    labels=train_data["label"],
    num_clients=numberOFClients,
    alpha=0.1,          # strong label non-IID
    beta=None,          # balanced client sizes
    prior="uniform",
    seed=seed
)
print("sizes:", sizes, "min/max:", min(sizes), max(sizes))

In [ ]:
all_ids = np.concatenate([np.array(x) for x in client_idxs])
print("Assigned:", len(all_ids), "Unique:", len(np.unique(all_ids)))

In [ ]:
client_shards = make_client_shards(train_data, client_idxs)

print("Client sizes:", [len(c) for c in client_shards])
print("Min/Max:", min(len(c) for c in client_shards), max(len(c) for c in client_shards))
print("Example client[0]:", client_shards[0][0])


In [ ]:
train_proc = [ train_ds.map(preprocess_fn, batched=True, remove_columns=train_ds.column_names)
                                for train_ds in client_shards]
for i in train_proc:
        i.set_format(type="torch", columns=cols)

In [ ]:
train_loader = [ DataLoader(i,batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator) for i in train_proc]

# model realated function

In [ ]:
def base_model_loader(adapter_dir: str):
    base = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
        # Load LoRA adapter weights into base
    model = PeftModel.from_pretrained(base, adapter_dir, is_trainable=True)
    return model

In [ ]:
def _move_optimizer_state_to_device(opt_state, device):
    for state in opt_state.get("state", {}).values():
        for k, v in list(state.items()):
            if torch.is_tensor(v):
                state[k] = v.to(device, non_blocking=True)
            elif isinstance(v, (list, tuple)):
                state[k] = [t.to(device, non_blocking=True) if torch.is_tensor(t) else t for t in v]
            elif isinstance(v, dict):
                for kk, vv in list(v.items()):
                    if torch.is_tensor(vv):
                        v[kk] = vv.to(device, non_blocking=True)

In [ ]:
def load_adapter_state_dict(adapter_dir: str, map_location="cpu"):
    """
    Load adapter weights from a PEFT adapter directory.
    Returns a state_dict (CPU tensors).
    """
    safe_path = os.path.join(adapter_dir, ADAPTER_SAFE)
    bin_path  = os.path.join(adapter_dir, ADAPTER_BIN)

    if safe_load is not None and os.path.exists(safe_path):
        return safe_load(safe_path)
    elif os.path.exists(bin_path):
        return torch.load(bin_path, map_location=map_location)
    else:
        raise FileNotFoundError(f"No adapter weights found in {adapter_dir} ({ADAPTER_SAFE} or {ADAPTER_BIN}).")


def save_adapter_state_dict(state_dict, out_dir: str):
    """
    Save adapter weights to out_dir, prefer safetensors if available.
    """
    os.makedirs(out_dir, exist_ok=True)
    if safe_save is not None:
        safe_save(state_dict, os.path.join(out_dir, ADAPTER_SAFE))
    else:
        torch.save(state_dict, os.path.join(out_dir, ADAPTER_BIN))

In [ ]:
# ============================================================
# ===============  FEDLAW GPU + DISK-SAFE VERSION  ===========
# ============================================================

import torch
import torch.nn.functional as F
import gc, os

# ------------------------------------------------------------
# 1) Load all adapter state_dicts and move to GPU
# ------------------------------------------------------------
def load_all_states_to_device(adapter_dirs, device):
    """
    Loads each client's LoRA adapter state_dict and moves all tensors to GPU.
    """
    all_sds = []
    for d in adapter_dirs:
        sd = load_adapter_state_dict(d, map_location="cpu")
        # move tensors to GPU
        sd_gpu = {k: v.to(device).float() for k, v in sd.items()}
        all_sds.append(sd_gpu)
    return all_sds


# ------------------------------------------------------------
# 2) Fuse adapters on GPU: fused = gamma * Σ λ_i * θ_i
# ------------------------------------------------------------
def fuse_adapters(adapter_sds, lambdas, gamma):
    """
    adapter_sds : list of dict[k]=tensor_on_gpu
    lambdas     : tensor on GPU
    gamma       : float or tensor
    """
    if not torch.is_tensor(gamma):
        gamma = torch.tensor(gamma, device=lambdas.device)

    K = len(adapter_sds)
    fused = {}

    for key in adapter_sds[0]:
        acc = sum(lambdas[i] * adapter_sds[i][key] for i in range(K))
        fused[key] = gamma * acc

    return fused


# ------------------------------------------------------------
# 3) FedLAW Step 1: Train λ (client importance weights)
# ------------------------------------------------------------
def train_fedlaw_lambdas(
    adapter_sds,
    base_model_loader,
    global_adapter_dir,
    server_loader,
    device,
    lr=0.01,
    epochs=1
):
    K = len(adapter_sds)
    lambdas = torch.ones(K, device=device, requires_grad=True)
    optimizer = torch.optim.Adam([lambdas], lr=lr)

    for epoch in range(epochs):
        for batch in server_loader:

            optimizer.zero_grad()

            # softmax(λ)
            w = torch.softmax(lambdas, dim=0)

            # build fused adapter on GPU
            fused_sd = fuse_adapters(adapter_sds, w, gamma=1.0)

            # load base model + fused adapter
            fused_model = base_model_loader(global_adapter_dir).to(device)
            fused_model.load_state_dict(fused_sd, strict=False)

            # forward
            ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            loss = fused_model(ids, attention_mask=attn, labels=labels).loss

            loss.backward()
            optimizer.step()

            del fused_model
            torch.cuda.empty_cache()
            gc.collect()

    # return lambdas as python list
    return torch.softmax(lambdas, dim=0).detach().cpu().tolist()


# ------------------------------------------------------------
# 4) FedLAW Step 2: Train γ (global scaling factor)
# ------------------------------------------------------------
def train_fedlaw_gamma(
    adapter_sds,
    lambdas,                    # list from previous step
    base_model_loader,
    global_adapter_dir,
    server_loader,
    device,
    lr=0.01,
    epochs=1
):
    # convert lambdas → GPU tensor
    lambdas = torch.tensor(lambdas, device=device)

    gamma = torch.tensor(1.0, device=device, requires_grad=True)
    optimizer = torch.optim.Adam([gamma], lr=lr)

    for epoch in range(epochs):
        for batch in server_loader:

            optimizer.zero_grad()

            # fuse with current gamma
            fused_sd = fuse_adapters(adapter_sds, lambdas, gamma)

            fused_model = base_model_loader(global_adapter_dir).to(device)
            fused_model.load_state_dict(fused_sd, strict=False)

            ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            loss = fused_model(ids, attention_mask=attn, labels=labels).loss

            loss.backward()
            optimizer.step()

            del fused_model
            torch.cuda.empty_cache()
            gc.collect()

    return float(gamma.detach().cpu())

@torch.no_grad()
def average_adapter_states(adapter_dirs, weights=None):
    """
    Average LoRA adapter weights across clients on CPU.

    adapter_dirs: list of dirs (each contains adapter_model.bin/safetensors)
    weights: list of floats same length; if None => uniform

    Returns:
        avg_state_dict (CPU tensors)
    """
    n = len(adapter_dirs)
    if n == 0:
        raise ValueError("No adapter dirs provided.")

    if weights is None:
        weights = [1.0 / n] * n
    else:
        s = sum(weights)
        weights = [w / s for w in weights]

    # load first
    avg = load_adapter_state_dict(adapter_dirs[0], map_location="cpu")
    for k in avg:
        avg[k] = avg[k].float() * weights[0]

    # accumulate rest
    for w, d in zip(weights[1:], adapter_dirs[1:]):
        sd = load_adapter_state_dict(d, map_location="cpu")
        for k in avg:
            avg[k] += sd[k].float() * w

    # cast back to original dtype if you want (optional)
    return avg
# ------------------------------------------------------------
# 5) Fully Updated FedLAW Aggregation
# ------------------------------------------------------------
def aggregate_adapters(
    selected_adapter_dirs,
    algorithm: str,
    global_adapter_dir: str,
    train_loaders=None,
    client_ids=None,
    wagg_set=None,
    base_model_loader=None,
    device="cuda",
):
    """
    Supports: Sophia, FedAvg, wagg, fedlaw
    """

    # ============= SOPHIA (simple average) =================
    if algorithm == "Sophia":
        avg_sd = average_adapter_states(selected_adapter_dirs, weights=None)
        save_adapter_state_dict(avg_sd, global_adapter_dir)
        return

    # ============= FedAvg =================
    if algorithm == "FedAvg":
        assert train_loaders is not None and client_ids is not None
        sizes = [len(train_loaders[cid]) for cid in client_ids]
        total = sum(sizes)
        weights = [s / total for s in sizes]
        avg_sd = average_adapter_states(selected_adapter_dirs, weights=weights)
        save_adapter_state_dict(avg_sd, global_adapter_dir)
        return

    if algorithm == "wagg":

        assert wagg_set is not None and base_model_loader is not None

        # -----------------------------------------
        # 1. Compute GLOBAL model validation NLL
        # -----------------------------------------
        global_model = base_model_loader(adapter_dir=global_adapter_dir).to(device)
        global_model.eval()

        _, global_nll = evaluate(global_model, wagg_set, device)

        del global_model
        torch.cuda.empty_cache()
        gc.collect()

        # -----------------------------------------
        # 2. Compute CLIENT scores
        # -----------------------------------------
        scores = []

        for d in selected_adapter_dirs:

            m = base_model_loader(adapter_dir=d).to(device)
            m.eval()

            _, client_nll = evaluate(m, wagg_set, device)

            # score = improvement in NLL
            delta = max(0.0, global_nll - client_nll)
            scores.append(delta)

            del m
            torch.cuda.empty_cache()
            gc.collect()

        # -----------------------------------------
        # 3. Normalize with softmax (stable)
        # -----------------------------------------
        # scores_t = torch.tensor(scores, dtype=torch.float32)
        sum_scores = sum(scores)
        if scores_t.sum() == 0:
            # fallback: keep global model only
            selected_adapter_dirs[:] = [global_adapter_dir]
            weights = [1.0]
        else:
            weights = [s / sum_scores for s in scores]
            """max_s = scores_t.max()
            weights = torch.softmax(scores_t - max_s, dim=0).tolist()"""
        print("WAGG weights:", weights)

        # -----------------------------------------
        # 4. Final weighted aggregation of adapters
        # -----------------------------------------
        print(weights)
        avg_sd = average_adapter_states(selected_adapter_dirs, weights=weights)
        save_adapter_state_dict(avg_sd, global_adapter_dir)
        return

    # ============= FEDLAW =================
    if algorithm.lower() == "fedlaw":

        assert base_model_loader is not None
        assert wagg_set is not None  # proxy validation set

        # Load client adapters to GPU
        adapter_sds = load_all_states_to_device(selected_adapter_dirs, device)

        # ---- Step 1: learn lambdas ----
        lambdas = train_fedlaw_lambdas(
            adapter_sds,
            base_model_loader,
            global_adapter_dir,
            wagg_set,
            device=device,
            lr=0.01,
            epochs=2
        )

        # ---- Step 2: learn gamma ----
        gamma = train_fedlaw_gamma(
            adapter_sds,
            lambdas,
            base_model_loader,
            global_adapter_dir,
            wagg_set,
            device=device,
            lr=0.01,
            epochs=2
        )

        # ---- Step 3: final fuse ----
        fused_sd = fuse_adapters(
            adapter_sds,
            torch.tensor(lambdas, device=device),
            gamma
        )

        # ---- Save fused adapter to disk ----
        save_adapter_state_dict(
            {k: v.cpu() for k, v in fused_sd.items()},  # save on CPU
            global_adapter_dir
        )

        return

    raise ValueError(f"Unknown algorithm: {algorithm}")

In [ ]:
def get_client_state_paths(edge: int, base_state_dir="client_state"):
    state_dir = os.path.join(base_state_dir, f"edge_{edge}")
    os.makedirs(state_dir, exist_ok=True)
    opt_path = os.path.join(state_dir, "optimizer.pt")
    return state_dir, opt_path

def get_round_adapter_dir(round_id: int, edge: int, cache_root="round_cache"):
    d = os.path.join(cache_root, f"round_{round_id}", f"edge_{edge}")
    os.makedirs(d, exist_ok=True)
    return d

# train loop

In [ ]:
def train_firstorder(
    base_model_loader,
    global_adapter_dir: str,
    train_loaders,
    test_loader,
    valid_loaders=None,
    algorithm="Sophia",
    grad_clip=1.0,
    lr=3e-5,
    epochs=1,
    rounds=10,
    num_clients=10,
    clients_per_round=4,
    seed=42,
    device="cuda",
    k=10
):

    results = []

    for globalRound in range(25):
        print(f"\n##################  Round {globalRound}  ########################")

        chosen = get_trained_client(1, num_clients, globalRound)

        selected_adapter_dirs = []
        selected_client_ids = []

        for edge in range(num_clients):
            if edge not in chosen:
                continue

            print(f"\n################## edge {edge} ########################")

            out_adapter_dir = get_round_adapter_dir(globalRound, edge)
            state_dir, opt_path = get_client_state_paths(edge)

            # ---- load model (base + current global adapter) ----
            model = base_model_loader(adapter_dir=global_adapter_dir).to(device)
            model.train()

            params = [p for p in model.parameters() if p.requires_grad]

            new_lr = cosine_lr(initial_lr=lr,round_idx=globalRound,total_rounds=rounds,eta_min=8e-7)
            optimizer = torch.optim.Adam(
                params,
                lr=new_lr
            )

            if os.path.exists(opt_path):
                opt_state = torch.load(opt_path, map_location="cpu")
                _move_optimizer_state_to_device(opt_state, device)
                optimizer.load_state_dict(opt_state)
                print(f"Loaded optimizer state for edge {edge} from {opt_path}")

            for param_group in optimizer.param_groups:
                param_group["lr"] = new_lr


            # ---- local training ----
            for epoch in range(1, epochs + 1):
                running_loss = 0.0

                for step, batch in enumerate(train_loaders[edge], start=1):
                    input_ids = batch["input_ids"].to(device, non_blocking=True)
                    attention_mask = batch["attention_mask"].to(device, non_blocking=True)
                    labels = batch["labels"].to(device, non_blocking=True)

                    optimizer.zero_grad(set_to_none=True)

                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels
                    )
                    loss = outputs.loss
                    loss.backward()

                    optimizer.step()

                    running_loss += float(loss.detach().cpu())

                avg_train_loss = running_loss / max(1, len(train_loaders[edge]))
                val_acc, val_loss = evaluate(model, test_loader, device)
                print(f"Epoch {epoch}: train_loss={avg_train_loss:.4f}  val_acc={val_acc:.4f} val_loss={val_loss:.4f}")

            # ---- save adapter + optimizer state ----
            model.save_pretrained(out_adapter_dir)
            torch.save(optimizer.state_dict(), opt_path)

            selected_adapter_dirs.append(out_adapter_dir)
            selected_client_ids.append(edge)

            # ---- free GPU memory ----
            del model
            del optimizer
            torch.cuda.empty_cache()
            gc.collect()

        # ---- server aggregation ----
        aggregate_adapters(
            selected_adapter_dirs,
            algorithm=algorithm,
            global_adapter_dir=global_adapter_dir,
            train_loaders=train_loaders,
            client_ids=selected_client_ids,
            wagg_set=valid_loaders if valid_loaders is not None else None,
            base_model_loader=base_model_loader,
            device=device,
        )

        # ---- evaluate updated global model ----
        global_model = base_model_loader(adapter_dir=global_adapter_dir).to(device)
        test_acc, global_loss = evaluate(global_model, test_loader, device)

        results.append(test_acc)
        print(f"Round {globalRound}: global_acc={test_acc:.4f}  global_loss={global_loss:.4f}")

        del global_model
        torch.cuda.empty_cache()
        gc.collect()

        # ---- delete per-round cache ----
        round_dir = os.path.join("round_cache", f"round_{globalRound}")
        if os.path.exists(round_dir) and globalRound!=99:
            shutil.rmtree(round_dir, ignore_errors=True)

    return results

In [ ]:
del model
torch.cuda.empty_cache()
gc.collect()
start_time = time.time()  # Record start time
results = train_firstorder(
    base_model_loader,          # function(adapter_dir) -> model (base + adapter loaded)
    global_adapter_dir = Main_path,    # current global adapter checkpoint directory
    train_loaders = train_loader,
    test_loader = test_loader,
    valid_loaders= None ,         # if you use it elsewhere
    algorithm="Sophia",
    grad_clip=1.0,
    lr=LR,
    epochs=EPOCHS,
    rounds=rounds,
    num_clients=numberOFClients,
    clients_per_round=5,
    seed=seed,
    device=device)         # Call the function
end_time = time.time()  # Record end time

execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")

In [ ]:
for i in range(len(results)):
    print(f"round {i} = {results[i]}")